# Infra-Bench CLS — SatlasPretrain Sentinel-1 SwinB Linear Probe

Linear-probe evaluation of SatlasPretrain's Sentinel-1 backbone on the
Infra-Bench CLS 13-class benchmark. Frozen backbone, trainable linear
head, 3 seeds, spatial split.

## Model

- **SatlasPretrain `Sentinel1_SwinB_SI`** — Swin-B backbone pretrained on
  the SatlasPretrain multi-task S1 dataset (Bastani et al. 2023).
  <https://github.com/allenai/satlas>
- 2-channel native SAR input `[VH, VV]`.
- Feature dim: 1024 (Swin-B pooled).

## Input pipeline

### Band selection
Our `.npy` storage: `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]`
(indices 0–8, so `VV = idx 7`, `VH = idx 8`).

SatlasS1 expects `[VH, VV]`. **`S1_BAND_INDICES = [8, 7]`** — reversed at
load time to match SatlasPretrain's channel convention.

### Normalization
Sensor-appropriate SAR preprocessing: **clip raw dB to `[-25, 0]` then
rescale to `[0, 1]`**. Bypasses `percentile_normalize` (which is used for
the S2 path elsewhere). The dB range captures the operationally relevant
backscatter signal above the noise floor and is the natural physical unit
for SAR.

## Split (shared across all Infra-Bench CLS FMs)

Spatial block-based split loaded from
`data/spatial_split/asset_id_to_split_v1.parquet`. Blocks assign a whole
~0.5° spatial region to the same partition. Invariant across seeds and
across every FM.

## Training protocol

- **25 epochs**, batch 16, AdamW, LR **1e-3**
- Class-weighted CE, weights capped at 10×
- **Frozen backbone** (`requires_grad=False`). Head is
  `InfraBenchClassifier`: `Linear(1024 → 13)` with `Dropout(0.1)`.
- **3 seeds** (314, 271, 161) varying head init + DataLoader shuffle only
  (linear-probe-on-frozen-features convention; no augmentation).
- **Best-val checkpoint restored before the held-out test pass**.

### Self-contained dataset

`NpyS1Dataset` is defined inline. The notebook does not depend on the
canonical `NpyInfrastructureDataset` from the curation zip.

## Per-sector F1 definition

Macro-average of per-class F1s for classes in that sector, computed on
the FULL test set. Return schema: `{n, macro_f1, acc,
per_class_f1_in_sector}` per sector.

## Aggregate output

Combines the 3 seeds into `mean ± std` and `per_seed` arrays. Write is
gated by `set(SEEDS) == set(FULL_PROTOCOL_SEEDS)`.

## Outputs

- Per-seed: `results/fm_eval_satlas_s1_v2_spatial/satlas_s1_v2_seed{314,271,161}_results.json`
- Aggregate: `results/fm_eval_satlas_s1_v2_spatial/satlas_s1_v2_aggregate.json`
- Confusion matrix: `results/fm_eval_satlas_s1_v2_spatial/confusion_matrix_satlas_s1_v2_aggregate.png`

## Runtime

`SMOKE_ONLY=True` by default. Do not run this and the SatlasS2 notebook
concurrently in the same Colab runtime — they share `/content/datasets/`
and would race on extracts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


In [4]:
%%capture
!pip install -q satlaspretrain-models scikit-learn pyarrow


In [ ]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# only load_split_artifact is needed at training time. if the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


In [ ]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_satlas_s1_v2_spatial'
# the split artifact ships inside the code zip, so it resolves from the
# extracted repo. Drive stays a fallback for setups that still stage it there.
_drive_split = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
try:
    from curation.paths import SPLIT_ARTIFACT as _repo_split
    SPLIT_ARTIFACT_PATH = str(_repo_split) if _repo_split.exists() else _drive_split
except Exception:
    SPLIT_ARTIFACT_PATH = _drive_split
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':              'water.water_works',   # legacy manifest tag
    'water.water_works':                  'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# s1-specific: our .npy stores S1 as [VV, VH] at indices [7, 8]. SatlasPretrain S1
# expects [VH, VV] so we reverse the order at load time: band_indices=[8, 7].
S1_BAND_INDICES = [8, 7]

# S1 normalization: clip dB to [DB_MIN, DB_MAX] and rescale to [0, 1].
#
# methodology note: SatlasS1's dedicated SAR backbone is paired with
# standard SAR preprocessing (dB clip -> [0, 1]). CROMA's joint S1+S2
# encoder uses percentile normalization on both modalities for input
# distribution consistency within the joint encoder; here we follow
# SatlasS1's intended preprocessing pipeline. this is the established
# SAR ML convention — dB scaling is the natural physical unit, and
# clipping to [-25, 0] captures the operationally relevant signal range
# above the noise floor.
DB_MIN, DB_MAX = -25.0, 0.0

# training (same as CROMA v2 / AE v2 / Prithvi).
IMAGE_SIZE = 224
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0
SEEDS      = [314, 271, 161]
# the aggregate write below is gated on matching this exactly
FULL_PROTOCOL_SEEDS      = [314, 271, 161]
RUN_NAME_PREFIX = 'satlas_s1_v2'
CONFUSION_CMAP = 'Purples'
CONFUSION_TITLE_PREFIX = 'SatlasS1 v2 spatial'


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:       {OUTPUT_DIR}')
print(f'Split artifact:   {SPLIT_ARTIFACT_PATH}')
print(f'S1 band indices:  {S1_BAND_INDICES}  (VH, VV — reversed from .npy storage)')
print(f'S1 dB clip:       [{DB_MIN}, {DB_MAX}] -> [0, 1]')
print(f'Training seeds:   {SEEDS}')
print(f'Linear probe:     {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')


In [ ]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


class NpyS1Dataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects S1 bands [8, 7] = [VH, VV] (reversed from our storage)
    and applies dB clip [-25, 0] -> [0, 1] (sensor-appropriate SAR
    preprocessing, see config cell for methodology rationale)."""
    def __init__(self, dataset_root,
                 band_indices=S1_BAND_INDICES,
                 db_min=DB_MIN, db_max=DB_MAX,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.db_min = float(db_min)
        self.db_max = float(db_max)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)

        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'
        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at:
                dropped['no_label'] += 1; continue
            if at not in self.allowed:
                dropped['filtered_type'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  NpyS1Dataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]                                # (2, H, W) [VH, VV]
        arr = np.clip(arr, self.db_min, self.db_max)
        arr = (arr - self.db_min) / (self.db_max - self.db_min)            # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)
        return {'image': torch.from_numpy(img),
                'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']
        img = F.interpolate(img.unsqueeze(0),
                            size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = NpyS1Dataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


In [ ]:
# load the spatial split artifact and slice each cell into train/val/test
# SubsetViews accordingly.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles in source_datasets have no split assignment '
          '(likely AlphaEarth-missing; will be excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')
print(f'  total in splits: {len(train_global) + len(val_global) + len(test_global)}')


In [ ]:
# diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison. same logic as v1's
# stratified_split (per-(region, sector) cell, by-class, seed=42).
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed — matches CROMA v2 and AlphaEarth v2; the '
      'spatial split is the same; only the training data layer differs across FMs.)')


In [ ]:
import torch.nn as nn
import satlaspretrain_models as spm


class SatlasS1Backbone(nn.Module):
    """SatlasPretrain Sentinel-1 backbone — natively 2-channel.

    No first-conv adapter: the backbone's pretrained Conv2d is already
    2-channel (VH + VV in that order). Applying the v1 S2-style averager
    adapter to a 2->2 case would AVERAGE the pretrained VH-specific and
    VV-specific weights together, destroying polarization specialization.
    The dataset layer takes care of the channel reordering (VV, VH in
    our .npy storage -> VH, VV via S1_BAND_INDICES=[8, 7])."""
    NAME = 'satlas_pretrain_swinb_s1'
    FEATURE_DIM = 1024

    def __init__(self, freeze=True):
        super().__init__()
        weights_manager = spm.Weights()
        self.backbone = weights_manager.get_pretrained_model(
            model_identifier='Sentinel1_SwinB_SI', fpn=False,
        )
        self.feature_dim = self.FEATURE_DIM
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        feature_maps = self.backbone(x)
        last = feature_maps[-1]
        return last.mean(dim=[2, 3])


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


def BACKBONE_FACTORY(freeze=True):
    return SatlasS1Backbone(freeze=freeze)


print('SatlasS1Backbone + InfraBenchClassifier defined.')


In [ ]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """Per-sector F1 v2 — matches CROMA v2's `_per_sector_v2` exactly.
    Macro-average of per-class F1s for the classes in that sector,
    computed on the FULL test set. NOT a macro-F1 restricted to in-sector
    samples (that v1 definition was the buggy version corrected via
    `per_sector_f1_catchall.ipynb` post-hoc). Returns dict with
    `{'n', 'macro_f1', 'acc', 'per_class_f1_in_sector'}` per sector."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition). v1 used '
            'a grouped_f1 that filtered samples to in-sector before computing '
            'macro_f1 — that version is deprecated and not reported here.'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set):
    set_seed(seed)
    print(f'\n--- seed {seed} ---')
    backbone = BACKBONE_FACTORY(freeze=True)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=LP_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=LP_LR, weight_decay=1e-4)

    run_name = f'{RUN_NAME_PREFIX}_seed{seed}_linear_probe'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt  = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(LP_EPOCHS):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': LP_EPOCHS,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    # restore best-val weights BEFORE running the held-out test pass.
    # fixes the v1 methodological gap where test ran on final-epoch state.
    # =====================================================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     backbone.NAME,
        'condition':    'linear_probe',
        'num_epochs':   LP_EPOCHS,
        'seed':         seed,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).')


In [ ]:
# default SMOKE_ONLY=True. flip to False and re-run this cell + the next
# to start the 3-seed training run.
SMOKE_ONLY = False

print('Loading SatlasS1 backbone for smoke check...')
set_seed(SEEDS[0])
sb = SatlasS1Backbone(freeze=True)
sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
print(f'  feature_dim = {sb.feature_dim}')
print(f'  trainable params = {sum(p.numel() for p in sm.parameters() if p.requires_grad):,}')

smoke_loader = DataLoader(train_global, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
img = batch['image']
print(f'  Batch shape: {tuple(img.shape)}  (expect [4, 2, {IMAGE_SIZE}, {IMAGE_SIZE}])')
print(f'  Batch range: [{img.min().item():.4f}, {img.max().item():.4f}]  (expect ~[0, 1])')
sm.eval()
with torch.no_grad():
    logits = sm(img.to(DEVICE))
print(f'  Logits shape: {tuple(logits.shape)}, range [{logits.min().item():.4f}, '
      f'{logits.max().item():.4f}], finite={torch.isfinite(logits).all().item()}')

print(f'\nSmoke OK. SMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


In [ ]:
# ============================================================================
# multi-seed invocation. trains 3 seeds sequentially, saves per-seed JSON,
# computes aggregate stats, writes confusion-matrix PNG. gated by SMOKE_ONLY.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping all training. Set False and re-run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    # ---- aggregate stats with per_seed arrays ---------------------------
    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {
            'mean':     float(arr.mean()),
            'std':      float(arr.std(ddof=0)),
            'per_seed': [float(v) for v in arr],
        }

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {
            'class':    CLASS_NAMES[i],
            'idx':      i,
            'mean_f1':  float(per_class_arr[:, i].mean()),
            'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
            'per_seed': [float(v) for v in per_class_arr[:, i]],
        }
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s_per_seed = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1']
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s_per_seed)),
            'std_macro_f1':  float(np.std(f1s_per_seed, ddof=0)),
            'per_seed':      [float(v) for v in f1s_per_seed],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s_per_seed = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s_per_seed)),
            'std_macro_f1':  float(np.nanstd(f1s_per_seed, ddof=0)),
            'per_seed':      [float(v) for v in f1s_per_seed],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    # a partial rerun must not clobber a complete three-seed aggregate
    if set(SEEDS) == set(FULL_PROTOCOL_SEEDS):
        agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol '
              f'{FULL_PROTOCOL_SEEDS}. Skipping the aggregate write '
              f'so any existing three-seed aggregate survives.')

    # ---- aggregate confusion matrix (sum across seeds, row-normalized) --
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm),
                         where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap=CONFUSION_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'{CONFUSION_TITLE_PREFIX} — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    # ---- summary print ---------------------------------------------------
    print('\n' + '=' * 76)
    print(f'{CONFUSION_TITLE_PREFIX} — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
